[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_migration_mcp_demo.ipynb)

# BigQuery Migration Demo - Managed MCP

**WORKING VERSION - Tested April 2026**

This notebook demonstrates:
1. SQL translation using BQMS MCP
2. Data transfer using DTS MCP

## Key Technical Details

**Location Handling:**
- BQMS translate_query requires 'us' or 'eu' multi-region (NOT regional like 'us-central1')
- DTS supports regional locations ('us-central1')
- Gemini 3.1 Pro Preview requires 'global' location
- Solution: Lock each service location via URL query parameters

**Flow:**
1. Empty BigQuery table created
2. JSONL data uploaded to GCS (simulates Hive external table)
3. Agent translates HiveQL → GoogleSQL using BQMS
4. Agent transfers data GCS → BigQuery using DTS
5. Verify data arrival and test translated query

**Prerequisites:**
- BQMS and DTS APIs enabled
- IAM roles: mcp.toolUser, bigquerymigration.editor, bigquery.admin, iam.serviceAccountUser
- Service account for DTS execution

In [ ]:
# Install dependencies
%pip install "google-adk>=1.28.0" google-genai "google-cloud-bigquery[pandas]" google-cloud-storage db-dtypes nest-asyncio --quiet

# Authenticate
try:
    from google.colab import auth
    auth.authenticate_user()
    print('✅ Authenticated via Colab')
except ModuleNotFoundError:
    print('✅ Using Application Default Credentials')

In [ ]:
# Configuration
import os
import nest_asyncio
nest_asyncio.apply()

PROJECT_ID = 'YOUR_PROJECT_ID'  # @param {type:"string"}
SERVICE_ACCOUNT = 'YOUR_SERVICE_ACCOUNT'  # @param {type:"string"}
LOCATION = 'us-central1'  # @param {type:"string"}

# CRITICAL: Environment variables must align with service locations
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

# Dataset/table config
DATASET = "migration_demo"
TABLE = "user_events"
BUCKET = f"{PROJECT_ID}-hive-data"

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Target: {PROJECT_ID}.{DATASET}.{TABLE}")
print(f"Source: gs://{BUCKET}/data.json")

In [ ]:
# Enable APIs
!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable bigquery.googleapis.com bigquerymigration.googleapis.com bigquerydatatransfer.googleapis.com storage.googleapis.com --quiet
!gcloud beta services mcp enable bigquerymigration.googleapis.com --quiet
!gcloud beta services mcp enable bigquerydatatransfer.googleapis.com --quiet
print("✅ APIs enabled")

In [ ]:
# Grant IAM roles
import time

roles = [
    "roles/mcp.toolUser",
    "roles/bigquerymigration.editor",
    "roles/bigquery.admin",
    "roles/iam.serviceAccountUser"
]

prefix = "serviceAccount" if "gserviceaccount.com" in SERVICE_ACCOUNT else "user"
member = f"{prefix}:{SERVICE_ACCOUNT}"

for role in roles:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} --member={member} --role={role} --condition=None --quiet > /dev/null

print("✅ IAM roles granted")
print("Waiting 60s for propagation...")
time.sleep(60)
print("✅ Ready")

In [ ]:
# Setup: Create empty BQ table + upload JSONL to GCS
from google.cloud import bigquery, storage
from google.api_core import exceptions
import json

# Create empty BigQuery table
bq = bigquery.Client(project=PROJECT_ID, location='us-central1')

dataset_ref = bq.create_dataset(f"{PROJECT_ID}.{DATASET}", exists_ok=True)

schema = [
    bigquery.SchemaField("user_id", "INTEGER"),
    bigquery.SchemaField("session_id", "STRING"),
    bigquery.SchemaField("event_types", "STRING", mode="REPEATED"),
    bigquery.SchemaField("product_ids", "INTEGER", mode="REPEATED"),
    bigquery.SchemaField("dt", "DATE"),
]

table_ref = bigquery.Table(f"{PROJECT_ID}.{DATASET}.{TABLE}", schema=schema)
bq.create_table(table_ref, exists_ok=True)
bq.query(f"TRUNCATE TABLE `{PROJECT_ID}.{DATASET}.{TABLE}`").result()

print(f"✅ Empty table created: {PROJECT_ID}.{DATASET}.{TABLE}")

# Upload JSONL data to GCS
gcs = storage.Client(project=PROJECT_ID)

try:
    bucket = gcs.create_bucket(BUCKET, location='us-central1')
except exceptions.Conflict:
    bucket = gcs.get_bucket(BUCKET)

data = [
    {"user_id": 1001, "session_id": "s1", "event_types": ["view", "cart", "buy"], "product_ids": [101, 102], "dt": "2025-01-15"},
    {"user_id": 1002, "session_id": "s2", "event_types": ["view", "search"], "product_ids": [201], "dt": "2025-01-16"},
    {"user_id": 1003, "session_id": "s3", "event_types": ["view", "cart"], "product_ids": [301, 302], "dt": "2025-01-17"},
]

jsonl = '\n'.join([json.dumps(row) for row in data])
blob = bucket.blob('data.json')
blob.upload_from_string(jsonl, content_type='application/json')

print(f"✅ Uploaded {len(data)} rows to gs://{BUCKET}/data.json")

In [ ]:
# Initialize MCP toolsets
import subprocess

token = subprocess.check_output(['gcloud', 'auth', 'print-access-token']).decode('utf-8').strip()

headers = {
    "Authorization": f"Bearer {token}",
    "x-goog-user-project": PROJECT_ID
}

# CRITICAL: Lock locations in URL query params (overrides environment variable)
# BQMS translate_query ONLY accepts 'us' or 'eu', NOT regional locations
# This avoids env var conflicts when agent model needs different location
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

bqms = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"https://bigquerymigration.googleapis.com/mcp?project={PROJECT_ID}&location=us",
        headers=headers,
        timeout=120.0
    )
)

dts = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f"https://bigquerydatatransfer.googleapis.com/mcp?project={PROJECT_ID}&location={LOCATION}",
        headers=headers,
        timeout=120.0
    )
)

print(f"✅ BQMS MCP initialized (location='us' via URL)")
print(f"✅ DTS MCP initialized (location='{LOCATION}' via URL)")

In [ ]:
# Create agent
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService

os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

agent = Agent(
    model="gemini-3.1-pro-preview",
    name="MigrationAgent",
    instruction="You are a migration expert. Complete tasks sequentially: translate SQL first, then transfer data.",
    tools=[bqms, dts]
)

runner = Runner(
    agent=agent,
    session_service=InMemorySessionService(),
    app_name="migration_demo",
    auto_create_session=True
)

print("✅ Agent ready")

In [ ]:
# Run migration
from google.genai import types

hiveql = """
SELECT
  user_id,
  COLLECT_SET(product_id) as products
FROM (
  SELECT user_id, product_id
  FROM user_events
  LATERAL VIEW EXPLODE(product_ids) pt AS product_id
  WHERE dt > '2025-01-01'
) t
GROUP BY user_id
"""

prompt = f"""
TASK 1: Translate this HiveQL to GoogleSQL:
{hiveql}

TASK 2: Transfer data from gs://{BUCKET}/data.json to {PROJECT_ID}.{DATASET}.{TABLE}
- Format: JSONL
- Service account: {SERVICE_ACCOUNT}
- Create transfer config and trigger run immediately

Complete both tasks in order.
"""

async def run():
    print("Starting migration...\n")
    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    
    async for event in runner.run_async(
        user_id="demo",
        session_id="s1",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"🤖 {part.text}")
                if part.function_call:
                    print(f"[TOOL] {part.function_call.name}")

await run()
print("\n✅ Migration complete")

In [ ]:
# Verify
import time
print("Waiting 30s for data transfer...")
time.sleep(30)

result = bq.query(f"SELECT COUNT(*) as cnt FROM `{PROJECT_ID}.{DATASET}.{TABLE}`").result()
count = next(result).cnt

if count > 0:
    print(f"✅ Success! {count} rows loaded")
    df = bq.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{TABLE}` LIMIT 5").to_dataframe()
    display(df)
else:
    print("❌ No data. Check DTS console or wait longer.")